# Volume 4. Claims And Experiments

Question: how do you tell the difference between a demo, an experiment, a claim, and a literature-facing result?

This notebook is a map of the research workflow. It does not rerun long experiments; it reads the repository indexes and turns them into inspection tables.


In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "research" / "registry.json").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Could not locate repository root")
    REPO_ROOT = REPO_ROOT.parent


The registry is the experiment suite inventory: where scripts live, where results should land, and what each suite is for.


In [ ]:
registry = json.loads((REPO_ROOT / "research" / "registry.json").read_text())
suite_rows = []
for suite in registry["suites"]:
    suite_rows.append({
        "suite": suite["name"],
        "status": suite["status"],
        "entry_points": len(suite.get("entry_points", [])),
        "experiments_dir": suite["experiments_dir"],
        "results_dir": suite["results_dir"],
        "focus": suite["focus"],
    })
suites = pd.DataFrame(suite_rows)
suites


In [ ]:
status_counts = Counter(suites["status"])
fig, ax = plt.subplots(figsize=(5.2, 3.0))
bars = ax.bar(status_counts.keys(), status_counts.values(), color=["#4d9de0", "#f2c14e"])
ax.bar_label(bars, padding=3)
ax.set_ylabel("suite count")
ax.set_title("Experiment suite status")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()
plt.close(fig)


The claims index separates formal claims from evidence summaries. This is the place to check before promoting a notebook observation into documentation.


In [ ]:
claims_index = json.loads((REPO_ROOT / "research" / "claims" / "index.json").read_text())
claims = pd.DataFrame([
    {
        "id": entry["id"],
        "status": entry["status"],
        "category": entry["category"],
        "scripts": len(entry.get("experiment_scripts", [])),
        "results": len(entry.get("result_artifacts", [])),
        "primary_artifact": entry.get("primary_artifact"),
        "title": entry["title"],
    }
    for entry in claims_index["entries"]
])
claims


In [ ]:
category_counts = claims.groupby(["category", "status"]).size().unstack(fill_value=0)
ax = category_counts.plot(kind="bar", stacked=True, figsize=(7.0, 3.5), color=["#59a14f", "#7768ae"])
ax.set_ylabel("claim or evidence entries")
ax.set_title("Claim inventory by category")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.xticks(rotation=30, ha="right")
plt.show()
plt.close(ax.figure)


Core questions are a question-first navigation layer. They point back to claims, experiment scripts, and result artifacts without pretending that every result is final.


In [ ]:
questions_index = json.loads((REPO_ROOT / "research" / "core_questions" / "index.json").read_text())
questions = pd.DataFrame([
    {
        "id": entry["id"],
        "status": entry["status"],
        "claim_refs": len(entry.get("claim_index_refs", [])),
        "scripts": len(entry.get("experiment_scripts", [])),
        "results": len(entry.get("result_artifacts", [])),
        "title": entry["title"],
    }
    for entry in questions_index["entries"]
])
questions


In [ ]:
coverage = pd.DataFrame([
    {
        "inventory": "experiment suites",
        "entries": len(suites),
        "active_or_formal": int((suites["status"] == "active").sum()),
    },
    {
        "inventory": "claims index",
        "entries": len(claims),
        "active_or_formal": int((claims["status"] == "formalized_claim").sum()),
    },
    {
        "inventory": "core questions",
        "entries": len(questions),
        "active_or_formal": int((questions["status"] == "curated_question").sum()),
    },
])
coverage


This volume gives the rule of thumb for the rest of the curriculum: demos can teach an operation, experiments can measure a regime, evidence summaries can organize results, and formal claims need artifacts, limitations, and falsification criteria.
